<a href="https://colab.research.google.com/github/konnitiha0816/-/blob/main/java%20yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yt-dlp deep-translator flask-cors
import yt_dlp
from flask import Flask, request, jsonify, render_template_string, send_file
from flask_cors import CORS
from deep_translator import GoogleTranslator
import threading
import socket
import os
from google.colab import output

# フォルダと基本設定
if not os.path.exists('static'): os.makedirs('static')
app = Flask(__name__, static_folder='static')
CORS(app)
translator = GoogleTranslator(source='auto', target='ja')

def translate_text(text):
    try:
        if not text: return ""
        return translator.translate(text)
    except: return text

# --- UIデザイン (YouTube完全参考 究極進化版7・DL高速化＆UI改善) ---
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang='ja'>
<head>
    <meta charset='UTF-8'>
    <title>java yt</title>
    <style>
        :root { --bg: #0f0f0f; --text: #fff; --gray: #aaaaaa; --hover: #272727; --primary: #cc0000; --border: #333; }
        body { font-family: 'Roboto', Arial, sans-serif; background-color: var(--bg); color: var(--text); margin: 0; display: flex; flex-direction: column; height: 100vh; overflow: hidden; }

        /* ヘッダー */
        header { height: 60px; padding: 0 16px; display: flex; align-items: center; justify-content: space-between; background: var(--bg); z-index: 100; flex-shrink: 0; }
        .header-left { display: flex; align-items: center; gap: 16px; }
        .menu-btn { background: none; border: none; color: white; font-size: 24px; cursor: pointer; padding: 8px; border-radius: 50%; transition: 0.2s; }
        .menu-btn:hover { background: var(--hover); }
        .logo { color: var(--primary); font-size: 22px; font-weight: bold; cursor: pointer; display: flex; align-items: center; gap: 4px; }

        .search-container { flex: 0 1 600px; display: flex; }
        .search-container input { width: 100%; background: #121212; border: 1px solid var(--border); color: white; padding: 10px 16px; border-radius: 20px 0 0 20px; outline: none; font-size: 16px; }
        .search-container input:focus { border-color: #555; }
        .search-container button { background: #222; border: 1px solid var(--border); border-left: none; padding: 0 20px; border-radius: 0 20px 20px 0; cursor: pointer; color: white; transition: 0.2s; font-size: 18px; }
        .search-container button:hover { background: #333; }

        #clock { font-size: 20px; font-weight: bold; font-family: 'Courier New', Courier, monospace; letter-spacing: 1px; color: var(--text); padding-right: 10px; }

        .wrapper { display: flex; flex: 1; overflow: hidden; position: relative; }

        /* オーバーレイ サイドバー */
        #navOverlay { display: none; position: fixed; inset: 0; background: rgba(0,0,0,0.5); z-index: 998; }
        nav { position: fixed; top: 60px; left: -250px; width: 240px; height: calc(100vh - 60px); background: var(--bg); padding: 12px; border-right: 1px solid var(--border); overflow-y: auto; z-index: 999; transition: left 0.3s ease; box-sizing: border-box; }
        nav.open { left: 0; }
        nav ul { list-style: none; padding: 0; margin: 0; }
        nav li { padding: 12px 16px; border-radius: 10px; cursor: pointer; transition: 0.2s; font-size: 15px; display: flex; align-items: center; gap: 16px; margin-bottom: 4px; }
        nav li:hover, nav li.active { background: var(--hover); }

        main { flex: 1; overflow-y: auto; padding: 24px; background: var(--bg); scroll-behavior: smooth; position: relative; }

        /* 絶対中央ローディング */
        #loadingView { display: none; position: absolute; inset: 0; background: var(--bg); z-index: 900; align-items: center; justify-content: center; flex-direction: column; }
        .spinner { width: 50px; height: 50px; border: 5px solid #333; border-top-color: var(--primary); border-radius: 50%; animation: spin 1s linear infinite; margin-bottom: 20px; }
        @keyframes spin { 100% { transform: rotate(360deg); } }

        /* サムネイルと時間表示の共通設定 */
        .thumb-container { position: relative; display: inline-block; width: 100%; }
        .duration-badge { position: absolute; bottom: 4px; right: 4px; background-color: rgba(0, 0, 0, 0.8); color: white; font-size: 12px; font-weight: 500; padding: 3px 4px; border-radius: 4px; line-height: 1; pointer-events: none; }

        /* ホーム・検索結果 */
        #home-greeting { font-size: 24px; margin-top: 0; margin-bottom: 20px; color: var(--text); }
        #home-title-prefix { font-size: 22px; font-weight: normal; margin-top: 0; margin-bottom: 24px; border-bottom: 1px solid var(--border); padding-bottom: 10px; }

        /* ショート動画 */
        .shorts-section { margin-bottom: 30px; }
        .shorts-section h3 { margin: 0 0 15px 0; font-size: 18px; display: flex; align-items: center; gap: 8px; font-weight:bold; }
        .shorts-grid { display: flex; overflow-x: auto; gap: 10px; padding-bottom: 10px; scrollbar-width: thin; }
        .shorts-card { flex: 0 0 110px; cursor: pointer; transition: transform 0.2s; }
        .shorts-card:hover { transform: translateY(-4px); }
        .shorts-card .thumb-container img { width: 100%; aspect-ratio: 9/16; border-radius: 8px; object-fit: cover; display: block; }
        .shorts-card h4 { font-size: 13px; margin: 6px 0 0 0; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; font-weight: 500; }

        /* 通常動画 */
        .video-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 20px 16px; margin-bottom: 30px; }
        .video-card { cursor: pointer; transition: transform 0.2s; display: flex; flex-direction: column; }
        .video-card:hover { transform: scale(1.02); }
        .video-card .thumb-container img { width: 100%; border-radius: 12px; aspect-ratio: 16/9; object-fit: cover; display: block; }
        .video-info { margin-top: 12px; display: flex; gap: 12px; }
        .video-info h3 { font-size: 16px; margin: 0 0 4px 0; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; font-weight: 500; }
        .video-info .channel-name { color: var(--gray); font-size: 14px; }

        /* プレイヤー画面 */
        #playerView { display: none; gap: 24px; align-items: flex-start; max-width: 1500px; margin: 0 auto; }
        @media (max-width: 1000px) { #playerView { flex-direction: column; } }

        #playerView.theater { flex-direction: column; max-width: 100%; padding: 0; }
        #playerView.theater .player-left { max-width: 100%; width: 100%; }
        #playerView.theater .player-right { max-width: 100%; width: 100%; padding: 24px; }

        .player-left { flex: 1; min-width: 0; width: 100%; transition: all 0.3s; }

        .player-container { position: relative; width: 100%; aspect-ratio: 16/9; border-radius: 12px; overflow: hidden; background: #000; box-shadow: 0 4px 15px rgba(0,0,0,0.5); }
        #playerLoading { position: absolute; inset: 0; background: #000; display: flex; flex-direction: column; align-items: center; justify-content: center; z-index: 10; color: white; display: none; }
        video { width: 100%; height: 100%; outline: none; }

        .video-title-area { margin-top: 16px; }
        .video-title-area h2 { font-size: 20px; margin: 0 0 12px 0; word-break: break-all; font-weight: bold; }

        /* アクションパネル */
        .action-panel { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; margin-bottom: 16px; gap: 10px; }
        .channel-info { display: flex; align-items: center; gap: 12px; }
        .channel-icon { width: 40px; height: 40px; border-radius: 50%; background: #555; display: flex; align-items: center; justify-content: center; font-size: 18px; overflow: hidden; }
        .channel-icon img { width: 100%; height: 100%; object-fit: cover; }
        .channel-text { font-weight: bold; font-size: 16px; }

        .controls-group { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
        .action-btn { background: var(--hover); border: none; color: white; padding: 8px 16px; border-radius: 20px; cursor: pointer; font-size: 14px; display: flex; align-items: center; gap: 6px; font-weight: 500; transition: 0.2s; }
        .action-btn:hover { background: #3f3f3f; }
        .action-btn.active { background: #ffebee; color: var(--primary); }

        .toggle-label { display: flex; align-items: center; gap: 6px; cursor: pointer; background: var(--hover); padding: 8px 14px; border-radius: 20px; font-size: 14px; transition: 0.2s; }
        .toggle-label:hover { background: #3f3f3f; }
        .toggle-label input { margin: 0; cursor: pointer; }
        .toggle-label input:disabled { cursor: not-allowed; }
        .toggle-label.disabled { opacity: 0.5; cursor: not-allowed; }

        #dlFormat option { color: black; background: white; }

        .desc-container { background: var(--hover); border-radius: 12px; padding: 12px; font-size: 14px; line-height: 1.5; margin-bottom: 24px; }
        .desc-text { display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical; overflow: hidden; white-space: pre-wrap; font-family: inherit; }
        .desc-text.expanded { -webkit-line-clamp: unset; }
        .toggle-btn { background: none; border: none; color: var(--text); font-weight: bold; cursor: pointer; padding: 0; margin-top: 8px; font-size: 14px; }

        .comments-section { margin-top: 24px; }
        .comments-section h3 { margin-bottom: 16px; font-size: 18px; }
        .comments-wrapper { position: relative; max-height: 250px; overflow: hidden; transition: max-height 0.3s; }
        .comments-wrapper.expanded { max-height: none; }
        .comments-toggle-btn { display: block; width: 100%; text-align: center; padding: 10px; background: transparent; color: var(--text); border: none; cursor: pointer; border-top: 1px solid var(--border); margin-top: 10px; font-size: 14px; font-weight: bold; }
        .comments-toggle-btn:hover { background: var(--hover); border-radius: 8px; }

        .comment-item { display: flex; gap: 16px; margin-bottom: 16px; font-size: 14px; }
        .comment-icon { width: 40px; height: 40px; border-radius: 50%; background: #444; flex-shrink: 0; display: flex; align-items: center; justify-content: center; }
        .comment-content .author { font-weight: 500; margin-bottom: 4px; font-size: 13px; }
        .comment-content .text { white-space: pre-wrap; line-height: 1.4; color: #f1f1f1; }

        /* 右側：関連動画 */
        .player-right { width: 400px; flex-shrink: 0; display: flex; flex-direction: column; }
        .related-list { display: flex; flex-direction: column; gap: 8px; }
        .related-card { display: flex; gap: 8px; cursor: pointer; transition: 0.2s; }
        .related-card:hover { background: var(--hover); border-radius: 8px; }
        .related-card .thumb-container { width: 168px; height: 94px; flex-shrink: 0; }
        .related-card .thumb-container img { width: 100%; height: 100%; border-radius: 8px; object-fit: cover; display: block; }
        .related-info { flex: 1; font-size: 12px; padding: 4px 0; }
        .related-info h4 { margin: 0 0 4px 0; font-size: 14px; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; font-weight: 500; }
        .related-info .c-name { color: var(--gray); }

        .load-more-btn { width: 100%; padding: 10px; margin-top: 15px; background: transparent; border: 1px solid var(--border); color: white; border-radius: 20px; cursor: pointer; font-size: 14px; transition: 0.2s; }
        .load-more-btn:hover { background: var(--hover); }
        .load-more-btn:disabled { color: var(--gray); cursor: not-allowed; }

        /* リスト画面 */
        .list-card { display: flex; gap: 16px; margin-bottom: 16px; cursor: pointer; position: relative; padding: 10px; border-radius: 12px; transition: 0.2s; max-width: 800px; }
        .list-card:hover { background: var(--hover); }
        .list-card .thumb-container { width: 240px; aspect-ratio: 16/9; flex-shrink: 0; }
        .list-card .thumb-container img { width: 100%; height: 100%; border-radius: 8px; object-fit: cover; display: block; }
        .list-card-info { flex: 1; }
        .list-card-info h3 { font-size: 18px; margin: 0 0 8px 0; font-weight: 500; }
        .btn-remove { position: absolute; right: 15px; top: 15px; background: transparent; color: var(--gray); border: none; font-size: 24px; cursor: pointer; }
        .btn-remove:hover { color: #ff4444; }

        /* 認証画面 */
        #auth-overlay { position: fixed; inset: 0; background: var(--bg); z-index: 2000; display: flex; flex-direction: column; align-items: center; justify-content: center; }
    </style>
</head>
<body>
    <!-- 認証 -->
    <div id='auth-overlay'>
        <h2 id="auth-greeting" style="font-size: 24px; margin-bottom: 10px; color: var(--gray);"></h2>
        <h1 style='color:var(--primary); font-size: 48px; margin-top: 0; margin-bottom: 40px; display:flex; align-items:center; gap:10px;'>🌳 java yt</h1>
        <input type='password' id='pass' placeholder='パスワードを入力' onkeydown='if(event.key==="Enter") checkAuth()' style='padding:14px 24px; border-radius:30px; border:1px solid #333; background:#121212; color:#fff; width:280px; font-size:16px; outline:none; text-align:center;'>
        <button onclick='checkAuth()' style='margin-top:24px; padding:14px 48px; border-radius:30px; background:var(--primary); color:white; border:none; cursor:pointer; font-size:16px; font-weight:bold;'>ログイン</button>
    </div>

    <!-- ヘッダー -->
    <header>
        <div class="header-left">
            <button class="menu-btn" onclick="toggleSidebar()">≡</button>
            <div class='logo' onclick='resetToHome()'>🌳 java yt</div>
        </div>
        <div class='search-container'>
            <input type='text' id='kw' placeholder='検索' onkeydown='if(event.key==="Enter") startSearch()'>
            <button onclick='startSearch()'>🔍</button>
        </div>
        <div id="clock">00:00</div>
    </header>

    <div class='wrapper'>
        <div id="navOverlay" onclick="toggleSidebar()"></div>

        <nav id="sidebar">
            <ul>
                <li id="nav-home" onclick='goHome()'>🏠 ホーム</li>
                <hr style='border: none; border-top: 1px solid var(--border); margin: 12px 0;'>
                <li id="nav-history" onclick="showListView('history')">🕒 履歴</li>
                <li id="nav-favorites" onclick="showListView('favorites')">❤️ お気に入り</li>
            </ul>
        </nav>

        <main id="mainContent">
            <!-- ローディング画面 -->
            <div id="loadingView">
                <div class="spinner"></div>
                <h2 style="color:var(--text); font-size: 20px;">検索中...</h2>
            </div>

            <!-- ① ホーム/検索結果画面 -->
            <div id="homeView">
                <h2 id="home-greeting"></h2>
                <h3 id="home-title-prefix" style="display:none;"></h3>

                <div class="shorts-section" id="shortsContainer">
                    <h3>📱 ショート動画</h3>
                    <div id="homeShortsList" class="shorts-grid"></div>
                </div>

                <div id="homeVideoList" class="video-grid"></div>
                <button id="homeLoadMoreBtn" class="load-more-btn" style="max-width:300px; margin:0 auto; display:block;" onclick="loadMoreResults()">もっと見る</button>
            </div>

            <!-- ② プレイヤー画面 -->
            <div id="playerView">
                <div class="player-left">
                    <div class="player-container">
                        <div id="playerLoading">
                            <div class="spinner"></div>
                            <div style="font-size: 18px;">読み込み中...</div>
                        </div>
                        <video id="v" controls autoplay></video>
                    </div>

                    <div class="video-title-area">
                        <h2 id="vT">タイトル</h2>
                    </div>

                    <div class="action-panel">
                        <div class="channel-info">
                            <div class="channel-icon" id="vChannelIcon"></div>
                            <span id="vChannel" class="channel-text">チャンネル名</span>
                        </div>

                        <div class="controls-group">
                            <button id="favBtn" class="action-btn" onclick="toggleCurrentFav()">🤍 保存</button>
                            <label class="toggle-label"><input type='checkbox' id='loopToggle' onchange='updateLoop()'> ループ</label>
                            <label id="radioModeLabel" class="toggle-label" title="すぐに音声のみに切り替えます"><input type='checkbox' id='radioMode' onchange='handleRadioToggle()'> ラジオ</label>
                            <label class="toggle-label" title="タブを離れると停止"><input type='checkbox' id='autoPauseToggle' checked> 自動停止</label>

                            <div style="display:flex; background:var(--hover); border-radius:20px; overflow:hidden;">
                                <select id='dlFormat' style="background:transparent; color:white; border:none; padding:8px 10px; outline:none; cursor:pointer;">
                                    <option value='mp4'>MP4</option>
                                    <option value='mp3'>MP3</option>
                                </select>
                                <button id="dlBtnText" onclick='downloadCurrent()' style="background:transparent; border:none; color:white; border-left:1px solid #444; padding:8px 16px; cursor:pointer; transition:0.2s;">📥 DL</button>
                            </div>
                        </div>
                    </div>

                    <div class="desc-container">
                        <div id="vDesc" class="desc-text"></div>
                        <button id="toggleDesc" class="toggle-btn" onclick="toggleDescription()">もっと見る</button>
                    </div>

                    <div class="comments-section">
                        <h3>💬 コメント</h3>
                        <div class="comments-wrapper" id="commentsWrapper">
                            <div id="commentsList"></div>
                        </div>
                        <button id="toggleCommentsBtn" class="comments-toggle-btn" onclick="toggleComments()" style="display:none;">もっと見る</button>
                    </div>
                </div>

                <div class="player-right">
                    <h3 style="margin-top:0; margin-bottom:16px;">関連動画</h3>
                    <div id="relatedList" class="related-list"></div>
                    <button id="relatedLoadMoreBtn" class="load-more-btn" onclick="loadMoreRelated()">もっと見る</button>
                </div>
            </div>

            <!-- ③ リスト画面 -->
            <div id="listView" style="display: none;">
                <h2 id="listTitle" style="margin-top:0; margin-bottom:24px; font-size:24px;"></h2>
                <div id="listGrid"></div>
            </div>
        </main>
    </div>

    <script>
        // 時間を HH:MM:SS または MM:SS にフォーマットする関数
        function formatDuration(seconds) {
            if (!seconds) return '';
            const h = Math.floor(seconds / 3600);
            const m = Math.floor((seconds % 3600) / 60);
            const s = Math.floor(seconds % 60);
            if (h > 0) {
                return `${h}:${m.toString().padStart(2, '0')}:${s.toString().padStart(2, '0')}`;
            }
            return `${m}:${s.toString().padStart(2, '0')}`;
        }

        // --- 初期設定 ---
        const setFavicon = () => {
            const link = document.createElement('link');
            link.rel = 'icon';
            link.href = 'Copilot_20260304_131811.jpg';
            document.head.appendChild(link);
        };
        setFavicon();

        function updateClockAndGreeting() {
            const now = new Date();
            const h = now.getHours();
            const m = now.getMinutes().toString().padStart(2, '0');
            document.getElementById('clock').innerText = `${h}:${m}`;

            let greeting = "こんばんは";
            if(h >= 4 && h < 12) greeting = "おはようございます";
            else if(h >= 12 && h < 18) greeting = "こんにちは";

            document.getElementById('auth-greeting').innerText = greeting;
            document.getElementById('home-greeting').innerText = greeting;
        }
        setInterval(updateClockAndGreeting, 1000);
        updateClockAndGreeting();

        // --- 状態管理 ---
        const video = document.getElementById('v');
        let currentQuery = "", currentCount = 30;
        let relatedCount = 20, relatedVideoId = "", relatedFallbackQuery = "";
        let currentItem = null;
        let wasPlayingBeforeHidden = false;

        let seenVideoIds = new Set();
        let seenRelatedIds = new Set();

        if(!localStorage.getItem('history')) localStorage.setItem('history', '[]');
        if(!localStorage.getItem('favorites')) localStorage.setItem('favorites', '[]');

        function getStored(key) { return JSON.parse(localStorage.getItem(key)); }
        function setStored(key, data) { localStorage.setItem(key, JSON.stringify(data)); }

        async function api(path, body) {
            const res = await fetch(path, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(body)});
            return res.json();
        }

        function extractVideoId(urlOrId) {
            if(!urlOrId) return "";
            if(urlOrId.length === 11 && !urlOrId.includes('/')) return urlOrId;
            let match = urlOrId.match(/[?&]v=([^&]+)/);
            if(match) return match[1];
            match = urlOrId.match(/youtu\.be\/([^?]+)/);
            if(match) return match[1];
            match = urlOrId.match(/youtube\.com\/shorts\/([^?]+)/);
            if(match) return match[1];
            return urlOrId;
        }

        // --- UI切り替え ---
        function toggleSidebar() {
            const sidebar = document.getElementById('sidebar');
            const overlay = document.getElementById('navOverlay');
            sidebar.classList.toggle('open');
            overlay.classList.toggle('open');
        }

        function switchView(viewId) {
            document.getElementById('homeView').style.display = 'none';
            document.getElementById('playerView').style.display = 'none';
            document.getElementById('listView').style.display = 'none';
            document.getElementById('loadingView').style.display = 'none';

            document.getElementById(viewId).style.display = (viewId === 'playerView' || viewId === 'loadingView') ? 'flex' : 'block';

            document.querySelectorAll('nav li').forEach(li => li.classList.remove('active'));
            if(viewId === 'homeView') document.getElementById('nav-home').classList.add('active');

            document.getElementById('sidebar').classList.remove('open');
            document.getElementById('navOverlay').classList.remove('open');
            window.scrollTo(0,0);
        }

        function checkAuth() {
            if(document.getElementById('pass').value === "0816") {
                document.getElementById('auth-overlay').style.display = 'none';
                resetToHome();
            } else { alert("パスワードが違います"); }
        }

        // --- ホーム・検索 ---
        function goHome() {
            if(currentItem) {
                switchView('playerView');
                document.getElementById('nav-home').classList.add('active');
            } else { resetToHome(); }
        }

        function resetToHome() {
            currentItem = null;
            video.pause();
            document.getElementById('kw').value = '';
            document.getElementById('home-title-prefix').style.display = 'none';
            document.getElementById('home-greeting').style.display = 'block';

            seenVideoIds.clear();
            document.getElementById('homeShortsList').innerHTML = '';
            document.getElementById('homeVideoList').innerHTML = '';

            const bases = ["日本 人気", "トレンド 日本", "最新 エンタメ 日本", "おすすめ 日本", "話題 日本", "音楽 日本"];
            let baseQuery = bases[Math.floor(Math.random() * bases.length)];

            const hist = getStored('history'), favs = getStored('favorites');
            const combined = [...favs, ...hist];
            if(combined.length > 0) {
                const randomItem = combined[Math.floor(Math.random() * combined.length)];
                const titlePart = randomItem.title.split(" ")[0].substring(0, 8);
                baseQuery += " " + titlePart;
            }

            const randomStr = Math.random().toString(36).substring(7);
            currentQuery = baseQuery;
            currentCount = 30;

            switchView('loadingView');

            Promise.all([
                executeSearchOnly(baseQuery + " " + randomStr, currentCount, false),
                executeSearchOnly(baseQuery + " shorts", 30, true)
            ]).then(() => {
                switchView('homeView');
            });
        }

        function startSearch() {
            video.pause();
            const kw = document.getElementById('kw').value;
            if(!kw) return;
            document.getElementById('home-greeting').style.display = 'none';
            seenVideoIds.clear();
            document.getElementById('homeShortsList').innerHTML = '';
            document.getElementById('homeVideoList').innerHTML = '';

            switchView('loadingView');

            const prefixEl = document.getElementById('home-title-prefix');
            prefixEl.innerText = `検索結果：${kw}`;
            prefixEl.style.display = 'block';

            currentQuery = kw;
            currentCount = 30;

            Promise.all([
                executeSearchOnly(kw, currentCount, false),
                executeSearchOnly(kw + " shorts", 30, true)
            ]).then(() => {
                switchView('homeView');
            });
        }

        async function executeSearchOnly(query, count, isShortsSearch) {
            const items = await api('/api/search', {query: query, count: count});
            renderMixedResults(items, isShortsSearch);
        }

        function loadMoreResults() {
            const btn = document.getElementById('homeLoadMoreBtn');
            btn.innerText = "取得中...";
            btn.disabled = true;
            currentCount += 20;

            executeSearchOnly(currentQuery, currentCount, false).then(() => {
                btn.innerText = "もっと見る";
                btn.disabled = false;
            });
        }

        function renderMixedResults(items, forceShorts = false) {
            let shortsHtml = '', videoHtml = '';

            items.forEach(item => {
                const id = item.url || item.id;
                if(seenVideoIds.has(id)) return;
                seenVideoIds.add(id);

                const isShort = forceShorts || (item.duration && item.duration <= 65) || (item.title && item.title.toLowerCase().includes('short'));
                const escItem = JSON.stringify(item).replace(/\"/g, '&quot;');
                const thumb = item.thumbnails[0]?.url || '';
                const channel = item.channel || '';

                const durationHtml = item.duration ? `<div class="duration-badge">${formatDuration(item.duration)}</div>` : '';

                if(isShort) {
                    shortsHtml += `<div class='shorts-card' onclick="doPlay('${id}', ${escItem})">
                                    <div class="thumb-container">
                                        <img src='${thumb}' loading="lazy">
                                        ${durationHtml}
                                    </div>
                                    <h4>${item.title}</h4>
                                 </div>`;
                } else {
                    videoHtml += `<div class='video-card' onclick="doPlay('${id}', ${escItem})">
                                    <div class="thumb-container">
                                        <img src='${thumb}' loading="lazy">
                                        ${durationHtml}
                                    </div>
                                    <div class='video-info'>
                                        <div style="width: 100%;">
                                            <h3>${item.title}</h3>
                                            <div class='channel-name'>${channel}</div>
                                        </div>
                                    </div>
                                 </div>`;
                }
            });

            document.getElementById('homeShortsList').insertAdjacentHTML('beforeend', shortsHtml);
            document.getElementById('homeVideoList').insertAdjacentHTML('beforeend', videoHtml);

            document.getElementById('shortsContainer').style.display = (document.getElementById('homeShortsList').innerHTML.trim() === "") ? 'none' : 'block';
        }

        // --- リスト画面 (履歴/お気に入り) ---
        function showListView(type) {
            switchView('listView');
            const targetNav = document.getElementById(`nav-${type}`);
            if(targetNav) targetNav.classList.add('active');
            document.getElementById('listTitle').innerText = type === 'history' ? '🕒 視聴履歴' : '❤️ お気に入り';
            renderListCards(getStored(type), type);
        }

        function renderListCards(items, listType) {
            const container = document.getElementById('listGrid');
            if(items.length === 0) { container.innerHTML = '<div style="color:var(--gray); padding:20px;">動画がありません。</div>'; return; }
            let html = '';
            items.forEach((item) => {
                const id = item.url || item.id;
                const durationHtml = item.duration ? `<div class="duration-badge">${formatDuration(item.duration)}</div>` : '';

                html += `<div class='list-card' onclick="doPlay('${id}', ${JSON.stringify(item).replace(/\"/g, '&quot;')})">
                            <div class="thumb-container">
                                <img src='${item.thumbnails[0]?.url || ''}'>
                                ${durationHtml}
                            </div>
                            <div class='list-card-info'>
                                <h3>${item.title}</h3>
                                <div style="color:var(--gray); font-size:14px;">${item.channel || ''}</div>
                            </div>
                            <button class="btn-remove" onclick="removeListItem(event, '${id}', '${listType}')" title="削除">×</button>
                         </div>`;
            });
            container.innerHTML = html;
        }

        function removeListItem(event, id, type) {
            event.stopPropagation();
            let items = getStored(type);
            items = items.filter(i => (i.url||i.id) !== id);
            setStored(type, items);
            renderListCards(items, type);
        }

        // --- プレイヤー画面 ---
        async function doPlay(url, item) {
            currentItem = item;
            switchView('playerView');

            const radioCheckbox = document.getElementById('radioMode');
            const radioLabel = document.getElementById('radioModeLabel');
            radioCheckbox.checked = false;
            radioCheckbox.disabled = true;
            radioLabel.classList.add('disabled');

            video.src = "";
            document.getElementById('playerLoading').style.display = 'flex';

            document.getElementById('vT').innerText = item.title;
            document.getElementById('vChannel').innerText = item.channel || item.uploader || "チャンネル";
            document.getElementById('vChannelIcon').innerHTML = (item.channel || item.uploader || "C").charAt(0);
            document.getElementById('vDesc').innerText = "読み込み中...";
            document.getElementById('commentsList').innerHTML = "<div style='color:var(--gray);'>コメントを取得中...</div>";
            document.getElementById('vDesc').classList.remove('expanded');
            document.getElementById('toggleDesc').innerText = 'もっと見る';
            document.getElementById('commentsWrapper').classList.remove('expanded');
            document.getElementById('toggleCommentsBtn').style.display = 'none';

            checkFavState();

            let hist = getStored('history');
            hist = [item, ...hist.filter(i => (i.url||i.id) !== (item.url||item.id))].slice(0, 100);
            setStored('history', hist);

            const vId = extractVideoId(url || item.id);
            loadRelatedVideos(vId, item.title, item.channel || item.uploader || "", false);

            await loadStreamData(url);
        }

        async function loadStreamData(url, resumeTime = 0) {
            const isRadio = document.getElementById('radioMode').checked;
            const data = await api('/api/play', {url, is_radio: isRadio});

            video.src = data.stream_url + "?t=" + Date.now();
            if(resumeTime > 0) video.currentTime = resumeTime;

            const unlockRadio = () => {
                document.getElementById('radioMode').disabled = false;
                document.getElementById('radioModeLabel').classList.remove('disabled');
            };

            video.oncanplay = () => {
                document.getElementById('playerLoading').style.display = 'none';
                video.play();
                unlockRadio();
            };
            setTimeout(() => {
                document.getElementById('playerLoading').style.display = 'none';
                video.play();
                unlockRadio();
            }, 3000);

            updateLoop();

            if(data.uploader) {
                document.getElementById('vChannel').innerText = data.uploader;
                if(data.channel_icon) {
                    document.getElementById('vChannelIcon').innerHTML = `<img src="${data.channel_icon}">`;
                } else {
                    document.getElementById('vChannelIcon').innerHTML = data.uploader.charAt(0);
                }
            }
            if(data.description) document.getElementById('vDesc').innerText = data.description;

            let commentsHtml = '';
            if(data.comments && data.comments.length > 0) {
                data.comments.forEach(c => {
                    commentsHtml += `<div class="comment-item"><div class="comment-icon">${c.author.charAt(0)}</div>
                                     <div class="comment-content"><div class="author">${c.author}</div><div class="text">${c.text}</div></div></div>`;
                });
                document.getElementById('toggleCommentsBtn').style.display = data.comments.length > 3 ? 'block' : 'none';
                document.getElementById('toggleCommentsBtn').innerText = "もっと見る";
            } else {
                commentsHtml = "<div style='color:var(--gray);'>コメントはありません（または取得制限）。</div>";
            }
            document.getElementById('commentsList').innerHTML = commentsHtml;
        }

        function handleRadioToggle() {
            if(!currentItem) return;
            const radioCheckbox = document.getElementById('radioMode');
            const radioLabel = document.getElementById('radioModeLabel');
            radioCheckbox.disabled = true;
            radioLabel.classList.add('disabled');

            document.getElementById('playerLoading').style.display = 'flex';
            const currentTime = video.currentTime;
            loadStreamData(currentItem.url || currentItem.id, currentTime);
        }

        // --- 関連動画 ---
        async function loadRelatedVideos(videoId, title, channel, isAppend = false) {
            const btn = document.getElementById('relatedLoadMoreBtn');
            if(!isAppend) {
                relatedCount = 20;
                seenRelatedIds.clear();
                document.getElementById('relatedList').innerHTML = '';
                relatedVideoId = videoId;
                const shortTitle = title ? title.split(" ").slice(0, 2).join(" ") : "";
                relatedFallbackQuery = `${channel} ${shortTitle} 関連`;
            } else {
                relatedCount += 20;
                btn.innerText = "取得中...";
                btn.disabled = true;
            }

            const items = await api('/api/related', {
                video_id: relatedVideoId,
                fallback_query: relatedFallbackQuery,
                count: relatedCount
            });

            let html = '';
            items.forEach(item => {
                if(!item) return;
                const id = item.url || item.id;
                if(currentItem && (currentItem.url || currentItem.id) === id) return;
                if(seenRelatedIds.has(id)) return;
                seenRelatedIds.add(id);

                const thumb = (item.thumbnails && item.thumbnails.length > 0) ? item.thumbnails[0].url : `https://i.ytimg.com/vi/${id}/hqdefault.jpg`;
                const durationHtml = item.duration ? `<div class="duration-badge">${formatDuration(item.duration)}</div>` : '';

                html += `<div class='related-card' onclick="doPlay('${id}', ${JSON.stringify(item).replace(/\"/g, '&quot;')})">
                            <div class="thumb-container">
                                <img src='${thumb}' loading="lazy">
                                ${durationHtml}
                            </div>
                            <div class='related-info'><h4>${item.title}</h4><div class='c-name'>${item.channel || item.uploader || ''}</div></div>
                         </div>`;
            });

            document.getElementById('relatedList').insertAdjacentHTML('beforeend', html);

            if(isAppend) {
                btn.innerText = "もっと見る";
                btn.disabled = false;
            }
        }
        function loadMoreRelated() { loadRelatedVideos(relatedVideoId, "", "", true); }

        // --- アクション機能 ---
        function checkFavState() {
            if(!currentItem) return;
            const favs = getStored('favorites');
            const id = currentItem.url || currentItem.id;
            const isFav = favs.some(f => (f.url||f.id) === id);
            const btn = document.getElementById('favBtn');
            if(isFav) { btn.innerHTML = "❤️ 保存済み"; btn.classList.add('active'); }
            else { btn.innerHTML = "🤍 保存"; btn.classList.remove('active'); }
        }

        function toggleCurrentFav() {
            if(!currentItem) return;
            let favs = getStored('favorites');
            const id = currentItem.url || currentItem.id;
            const idx = favs.findIndex(x => (x.url||x.id) === id);
            if(idx > -1) favs.splice(idx, 1); else favs.unshift(currentItem);
            setStored('favorites', favs);
            checkFavState();
        }

        function toggleDescription() {
            const d = document.getElementById('vDesc');
            const b = document.getElementById('toggleDesc');
            if(d.classList.toggle('expanded')) b.innerText='一部を表示';
            else b.innerText='もっと見る';
        }

        function toggleComments() {
            const w = document.getElementById('commentsWrapper');
            const b = document.getElementById('toggleCommentsBtn');
            b.innerText = "取得中...";
            setTimeout(() => {
                if(w.classList.toggle('expanded')) b.innerText='一部を表示';
                else b.innerText='もっと見る';
            }, 300);
        }

        function updateLoop() { video.loop = document.getElementById('loopToggle').checked; }

        // DLボタンの処理（確実に待機し、UIを更新する）
        async function downloadCurrent() {
            if(!currentItem) return;
            const btn = document.getElementById('dlBtnText');

            // UIを処理中に変更して無効化
            btn.innerText = "処理中...";
            btn.disabled = true;
            btn.style.opacity = "0.5";
            btn.style.cursor = "not-allowed";

            const id = currentItem.url || currentItem.id;
            const fmt = document.getElementById('dlFormat').value;
            const url = `/api/download?url=${encodeURIComponent(id)}&format=${fmt}`;

            try {
                // ダウンロードが完了するまで待機
                const response = await fetch(url);
                if (!response.ok) throw new Error("Network response was not ok");

                // データを取得してブラウザ上で保存処理を実行
                const blob = await response.blob();
                const downloadUrl = window.URL.createObjectURL(blob);
                const a = document.createElement('a');
                a.href = downloadUrl;

                // ファイル名に使えない文字をアンダースコアに変換
                const safeTitle = (currentItem.title || 'video').replace(/[\\/:*?"<>|]/g, '_');
                a.download = `${safeTitle}.${fmt}`;

                document.body.appendChild(a);
                a.click();
                document.body.removeChild(a);
                window.URL.revokeObjectURL(downloadUrl);
            } catch(e) {
                console.error("Download error:", e);
                alert("ダウンロードに失敗しました。時間をおいて再試行してください。");
            } finally {
                // 処理が終わったら元の状態に戻す
                btn.innerText = "📥 DL";
                btn.disabled = false;
                btn.style.opacity = "1";
                btn.style.cursor = "pointer";
            }
        }

        // --- イベント・ショートカット ---
        document.addEventListener('visibilitychange', () => {
            if (document.getElementById('autoPauseToggle').checked) {
                if (document.hidden) { wasPlayingBeforeHidden = !video.paused; video.pause(); }
                else { if(wasPlayingBeforeHidden) video.play(); }
            }
        });

        let spacePressedTimer = null;
        window.addEventListener('keydown', (e) => {
            if (document.activeElement.tagName === 'INPUT') return;
            if (e.key === 'm' || e.key === 'M') video.muted = !video.muted;
            if (e.key === 'f' || e.key === 'F') {
                if(!document.fullscreenElement) { video.requestFullscreen().catch(()=>{}); }
                else { document.exitFullscreen(); }
            }
            if (e.key === 'j' || e.key === 'J') {
                document.getElementById('playerView').classList.toggle('theater');
            }
            if (e.key === 'ArrowRight') video.currentTime += 10;
            if (e.key === 'ArrowLeft') video.currentTime -= 10;
            if (e.key === ' ') {
                e.preventDefault();
                if (!spacePressedTimer) spacePressedTimer = setTimeout(() => { video.playbackRate = 2.0; }, 500);
            }
        });
        window.addEventListener('keyup', (e) => {
            if (e.key === ' ') {
                if (spacePressedTimer) {
                    clearTimeout(spacePressedTimer);
                    if (video.playbackRate === 2.0) video.playbackRate = 1.0;
                    else video.paused ? video.play() : video.pause();
                    spacePressedTimer = null;
                }
            }
        });
    </script>
</body>
</html>
"""

# --- バックエンド ---
@app.route('/')
def index(): return render_template_string(HTML_TEMPLATE)

@app.route('/api/search', methods=['POST'])
def search_api():
    query, count = request.json.get('query'), request.json.get('count', 30)
    ydl_opts = {'quiet':True, 'extract_flat':True, 'http_headers': {'Accept-Language': 'ja-JP,ja;q=0.9,en-US;q=0.8'}}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        results = ydl.extract_info(f"ytsearch{count}:{query}", download=False)['entries']
        return jsonify(results)

@app.route('/api/related', methods=['POST'])
def related_api():
    video_id = request.json.get('video_id')
    fallback_query = request.json.get('fallback_query')
    count = request.json.get('count', 20)

    ydl_opts = {'quiet':True, 'extract_flat':True, 'http_headers': {'Accept-Language': 'ja-JP,ja;q=0.9,en-US;q=0.8'}}

    if video_id:
        try:
            playlist_url = f"https://www.youtube.com/watch?v={video_id}&list=RD{video_id}"
            ydl_opts_pl = ydl_opts.copy()
            ydl_opts_pl['playlist_items'] = f'1-{count}'
            with yt_dlp.YoutubeDL(ydl_opts_pl) as ydl:
                info = ydl.extract_info(playlist_url, download=False)
                if 'entries' in info and info['entries']:
                    return jsonify(info['entries'])
        except Exception:
            pass

    if fallback_query:
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                results = ydl.extract_info(f"ytsearch{count}:{fallback_query}", download=False)['entries']
                return jsonify(results)
        except Exception:
            pass

    return jsonify([])

@app.route('/api/play', methods=['POST'])
def play_api():
    url, is_radio = request.json.get('url'), request.json.get('is_radio')
    path = 'static/video.mp4'
    if os.path.exists(path): os.remove(path)
    fmt = 'bestaudio/best' if is_radio else 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best'

    ydl_opts = {
        'format': fmt,
        'outtmpl': path,
        'merge_output_format': 'mp4',
        'quiet': True,
        'getcomments': True,
        'extractor_args': {'youtube': {'max_comments': ['15'], 'lang': ['ja']}},
        'http_headers': {'Accept-Language': 'ja-JP,ja;q=0.9,en-US;q=0.8'}
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

        channel_icon = None
        if 'channel_thumbnails' in info and info['channel_thumbnails']:
            channel_icon = info['channel_thumbnails'][-1]['url']

        comments_data = []
        if 'comments' in info and info['comments']:
            for c in info['comments'][:15]:
                comments_data.append({'author': c.get('author', 'ユーザー'), 'text': c.get('text', '')})

        return jsonify({
            'title': translate_text(info.get('title')),
            'description': translate_text(info.get('description')),
            'uploader': info.get('uploader', info.get('channel', 'Unknown Channel')),
            'channel_icon': channel_icon,
            'comments': comments_data,
            'stream_url': '/static/video.mp4'
        })

@app.route('/api/download')
def download_media():
    url, fmt = request.args.get('url'), request.args.get('format', 'mp4')
    ext = 'mp3' if fmt == 'mp3' else 'mp4'
    out = f'static/download.{ext}'

    if os.path.exists(out):
        try: os.remove(out)
        except: pass

    dl_fmt = 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best' if fmt == 'mp4' else 'bestaudio/best'

    # DL速度を向上させるために並列ダウンロードオプション(concurrent_fragment_downloads)を追加
    opts = {
        'format': dl_fmt,
        'outtmpl': out,
        'quiet': True,
        'concurrent_fragment_downloads': 5
    }

    if fmt == 'mp3':
        opts['postprocessors'] = [{'key': 'FFmpegExtractAudio','preferredcodec': 'mp3','preferredquality': '192'}]

    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    return send_file(out, as_attachment=True)

# --- サーバー起動 ---
def start_app():
    s = socket.socket(); s.bind(('', 0)); port = s.getsockname()[1]; s.close()
    threading.Thread(target=lambda: app.run(port=port), daemon=True).start()
    proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({port})')
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n✅ java yt 究極進化版7 (DL高速化＆UI改善) 起動完了！\n🔗 URL: {proxy_url}\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

start_app()

<>:360: SyntaxWarning: invalid escape sequence '\.'
<>:360: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_168/1645798543.py:360: SyntaxWarning: invalid escape sequence '\.'
  match = urlOrId.match(/youtu\.be\/([^?]+)/);


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:58195
INFO:werkzeug:Press CTRL+C to quit


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ java yt 究極進化版7 (DL高速化＆UI改善) 起動完了！
🔗 URL: https://58195-m-s-901yshn19jzn-b.us-east1-2.prod.colab.dev
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
